In [8]:
import pandas as pdsource 
import json
from kafka import KafkaConsumer
from neo4j import GraphDatabase

In [ ]:
KAFKA_TOPIC = 'transactions'
KAFKA_BOOTSTRAP_SERVERS = 'kafka_streaming_lab:9092'

NEO4J_CONFIG = {
    "uri": "bolt://neo4j_nosql_lab:7687", 
    "user": "neo4j",
    "password": "test1234"
}

driver = GraphDatabase.driver(NEO4J_CONFIG["uri"], auth=(NEO4J_CONFIG["user"], NEO4J_CONFIG["password"]))

consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    auto_offset_reset='earliest'
)

def process_neo4j(tx, data):
    # a) (User)-[:USES]->(Device) # Użytkownik używa urządzenia
    # b) (User)-[:TRANSFER]->(User) # Użytkownik A wykonał transfer do użytkownika B
    query = """
    // Tworzenie lub znalezienie użytkowników (nadawca i odbiorca)
    MERGE (s:User {name: $sender})
    MERGE (r:User {name: $receiver})
    
    // Tworzenie lub znalezienie urządzeń
    MERGE (ds:Device {device_id: $device_sender})
    MERGE (dr:Device {device_id: $device_receiver})
    
    // Punkt a) Użytkownicy używają urządzeń
    MERGE (s)-[:USES]->(ds)
    MERGE (r)-[:USES]->(dr)
    
    // Punkt b) Relacja transferu między użytkownikami
    CREATE (s)-[:TRANSFER { amount: $amount, timestamp: $timestamp,title: $title }]->(r)
    """
    tx.run(query, 
           sender=data['sender'],
           receiver=data['receiver'],
           device_sender=data['device_sender'],
           device_receiver=data['device_receiver'],
           amount=data['amount'],
           timestamp=data['timestamp'],
           title=data['title']
    )

print("Konsument Neo4j uruchomiony, czekam na transakcje...")

try:
    for message in consumer:
        tx_data = message.value
        
        with driver.session() as session:
            session.execute_write(process_neo4j, tx_data)
            
        print(f"Zmapowano w grafie: {tx_data['sender']} -> {tx_data['receiver']} ({tx_data['amount']} PLN)")

except Exception as e:
    print(f"Błąd podczas pracy konsumenta Neo4j: {e}")
finally:
    consumer.close()
    driver.close()


Konsument Neo4j uruchomiony, czekam na transakcje...
Zmapowano w grafie: user11 -> user12 (35.0 PLN)
Zmapowano w grafie: user13 -> user14 (60.0 PLN)
Zmapowano w grafie: user15 -> user16 (80.0 PLN)
Zmapowano w grafie: user1 -> user3 (45.0 PLN)
Zmapowano w grafie: user14 -> user16 (175.0 PLN)
Zmapowano w grafie: user15 -> user17 (185.0 PLN)
Zmapowano w grafie: user5 -> user6 (200.0 PLN)
Zmapowano w grafie: user7 -> user8 (300.0 PLN)
Zmapowano w grafie: user11 -> user12 (35.0 PLN)
Zmapowano w grafie: user15 -> user16 (80.0 PLN)
Zmapowano w grafie: user1 -> user3 (45.0 PLN)
Zmapowano w grafie: user2 -> user4 (55.0 PLN)
Zmapowano w grafie: user6 -> user8 (95.0 PLN)
Zmapowano w grafie: user7 -> user9 (105.0 PLN)
Zmapowano w grafie: user11 -> user13 (145.0 PLN)
Zmapowano w grafie: user12 -> user14 (155.0 PLN)
Zmapowano w grafie: user14 -> user16 (175.0 PLN)
Zmapowano w grafie: user15 -> user17 (185.0 PLN)
Zmapowano w grafie: user18 -> user20 (215.0 PLN)
Zmapowano w grafie: user3 -> user4 (75.